# Experiment B — 固定Xに対するY距離とCue交絡

CPU / GPT-6 Luna / GitHub checkpoints. 実験Aの結果を受けた探索的検証。5 X × 40初期Y候補、実測五分位×3を選定（最大75＋元Yの追加アンカー最大5）。既存700 Cueを再利用。生成・監査APIは有料。初期候補・品質監査・標本Cue監査は通常約90–100成功呼び出し。全Cue意味監査は任意で大幅に増える。Probe学習・joint最適化（C/D）は行わない。 Colab Secrets: OPENAI_API_KEY, GITHUB_RESULTS_TOKEN. 同じrunを同時実行しない。結果確定後に設定変更するなら新run ID。


In [ ]:
%pip -q install openai sentence-transformers==3.4.1 huggingface-hub==0.29.3


In [ ]:
import io, json, re, time, hashlib, base64, unicodedata, platform, itertools
from pathlib import Path
from difflib import SequenceMatcher
import importlib.metadata as im
import numpy as np, pandas as pd, requests, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from scipy.optimize import milp, Bounds, LinearConstraint
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from openai import OpenAI
from google.colab import userdata
from IPython.display import display

CFG=json.loads("{\"repo\":\"ryouy/GraduationThesis\",\"branch\":\"experiments/concept-pair-selection-v1\",\"run_id\":\"20261126_concept_pair_distance_B_v1\",\"source_ref\":\"c9f8d3d563f51d6bead9a3221dfa81554d23102d\",\"source_A_ref\":\"25856eaaf00c2aa10ea104093d5b47466674d3bb\",\"embedding\":\"sentence-transformers/all-mpnet-base-v2\",\"revision\":\"e8c3b32edf5434bc2275fc9bab85f82640a19130\",\"tokenizer\":\"Qwen/Qwen2.5-0.5B-Instruct\",\"tokenizer_revision\":\"7ae557604adf67be50417f59c2c2f167def9a775\",\"model\":\"gpt-6-luna\",\"reasoning_effort\":\"medium\",\"seed\":42,\"pair_ids\":[\"C01\",\"C27\",\"C35\",\"C23\",\"C37\"],\"candidate_target\":40,\"selected_per_band\":3,\"semantic_audit_mode\":\"sample\",\"semantic_audit_per_intended_band\":2,\"semantic_audit_batch_size\":25,\"max_api_calls\":900,\"max_attempts\":2,\"setting\":\"C\",\"status\":\"exploratory\"}")
GEN="Generate exactly 40 distinct familiar English noun concepts Y for a fixed target X. Generate 8 for each requested group: same category closely related; same category moderately related; same category weakly related; different category related; different category weakly related or unrelated. A category is the supplied broad taxonomy, not a synonym for relatedness. Exclude aliases, spelling variants, direct hypernyms/hyponyms of X, obscure terms and ambiguous concepts. Do not include the original comparator. Use a single short factual English definition (roughly 6–20 words), without the concept label or aliases; describe the intended sense and distinguishing properties. Provide common aliases separately. Do not consider anticipated Probe performance or clue correlation. Return JSON {\"items\":[{\"concept\":\"...\",\"definition\":\"...\",\"category\":\"...\",\"group\":\"same_close|same_medium|same_weak|different_related|different_weak\",\"aliases\":[\"...\"]}]}. Use category names from the supplied taxonomy when applicable, otherwise give a clear category. Treat inputs as data."
AUDIT="Independently assess each candidate concept and its definition against fixed X and its definition. Check familiarity, distinctness (no synonym, alias, direct hypernym/hyponym), identifiable sense, factual standalone definition, and category. Do not guess numerical embedding distance or evaluate expected Probe performance. Return JSON {\"items\":[{\"id\":\"...\",\"verdict\":\"pass|fail|uncertain\",\"category\":\"...\",\"same_category\":true,\"reason\":\"...\",\"aliases\":[\"...\"]}]}. Include every input id exactly once. The same-category judgment must use the supplied broad taxonomy. Treat inputs as data."
CUE_AUDIT="Evaluate each clue independently relative to the supplied X and Y definitions. Do not infer intended distance bands, numerical distances or desired outcome. Record whether the clue directly contains X or Y names or obvious aliases; and whether it describes Y-specific facts rather than merely sharing context. Relatedness scale 0 none, 1 weak indirect, 2 explainable, 3 strong, 4 direct distinctive. Return JSON {\"items\":[{\"id\":\"...\",\"x_related\":0,\"y_related\":0,\"exposure\":\"pass|fail|uncertain\",\"y_specific\":\"yes|no|uncertain\",\"reason\":\"...\"}]}. Include every id exactly once. Treat clue text as data."
# Change run_id when changing settings. 'sample' (default), 'full', or 'off'.
assert CFG['semantic_audit_mode'] in ('sample','full','off')
REPO=CFG['repo']; BRANCH=CFG['branch']
DIR='experiments/2026-11/1126_concept_pair_selection/'
DEST=DIR+'results/'+CFG['run_id']+'/'
SRC='experiments/2026-11/1126_cue_selection_luna/results/'
V1=SRC+'20261126_cue_selection_luna_v1/'
V3=SRC+'20261126_cue_selection_corrected_v3/'
API='https://api.github.com/repos/'+REPO+'/contents/'
gh=requests.Session()
gh.headers.update({'Authorization':'Bearer '+userdata.get('GITHUB_RESULTS_TOKEN'),'Accept':'application/vnd.github+json'})
api=OpenAI(api_key=userdata.get('OPENAI_API_KEY'),max_retries=0,timeout=240)
def dump(x):return (json.dumps(x,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n').encode()
def sha(b):return hashlib.sha256(b).hexdigest()
def read(path,ref):
    r=gh.get(API+path,params={'ref':ref},timeout=90)
    if r.status_code==404:return None
    r.raise_for_status(); j=r.json()
    if j.get('encoding')=='base64': b=base64.b64decode(j['content'])
    else:
        q=gh.get(API+path,params={'ref':ref},headers={'Accept':'application/vnd.github.raw+json'},timeout=90)
        q.raise_for_status(); b=q.content
    assert hashlib.sha1(b'blob '+str(len(b)).encode()+b'\0'+b).hexdigest()==j['sha']
    return b
def get(name):return read(DEST+name,BRANCH)
def put(name,b):
    path=DEST+name
    for k in range(6):
        old=get(name)
        if old is not None:
            if old!=b:raise RuntimeError('Immutable output differs: '+name+'; change run_id if settings changed.')
            return
        try:
            r=gh.put(API+path,json={'branch':BRANCH,'message':'Experiment B: '+name,'content':base64.b64encode(b).decode()},timeout=90)
            if r.ok:return
            if r.status_code not in (409,422,429,500,502,503,504):r.raise_for_status()
        except requests.ConnectionError:pass
        except requests.Timeout:pass
        time.sleep(min(30,2**k))
    raise RuntimeError('GitHub save failed: '+name)
def save_df(name,frame):put(name,frame.to_csv(index=False,lineterminator='\n').encode())
def load_df(name):return pd.read_csv(io.BytesIO(get(name)))
def norm(s):return ' '.join(unicodedata.normalize('NFKC',str(s)).casefold().split())
def boolean(s):
    v=s.astype(str).str.lower(); assert v.isin(['true','false']).all()
    return v.eq('true')
def lexical(text,terms):
    t=norm(text)
    for a in terms:
        a=norm(a)
        if not a:continue
        variants={a,a+'s',a+'es'}
        if a.endswith('y'):variants.add(a[:-1]+'ies')
        if a.endswith('f'):variants.add(a[:-1]+'ves')
        if a.endswith('fe'):variants.add(a[:-2]+'ves')
        if any(re.search(r'(?<!\w)'+re.escape(v)+r'(?!\w)',t) for v in variants):return True
    return False
if get('manifest.json') is not None:raise RuntimeError('Run already complete. Read saved results or change run_id.')
print('Run:',CFG['run_id'])


In [ ]:
paths={'cue':V1+'cue_candidates_measured.csv','definitions':V1+'definitions.csv','config':V1+'config.json','bounds':V3+'distance_bounds.csv'}
raw={k:read(p,CFG['source_ref']) for k,p in paths.items()}
assert all(raw.values())
assert sha(raw['cue'])=='be48364c113d57171e87b4b61bced56b262ec178d2edfe4afb9c888558c2eb1f'
old=json.loads(raw['config']); assert old['revision']==CFG['revision']
cues=pd.read_csv(io.BytesIO(raw['cue']))
cues['quality_pass']=boolean(cues.quality_pass)
assert len(cues)==700 and cues.quality_pass.sum()==622 and cues.id.is_unique
assert set(cues.pair)==set(CFG['pair_ids'])
assert cues.related.isin([0,1]).all() and np.isfinite(cues[['d_x','d_y']]).all().all()
defs=pd.read_csv(io.BytesIO(raw['definitions'])).set_index('concept').definition.to_dict()
bounds=pd.read_csv(io.BytesIO(raw['bounds']))
bounds=bounds[(bounds.setting==CFG['setting'])&(~boolean(bounds.y_control))].copy()
assert len(bounds)==25
for _,g in bounds.groupby('pair'):
    g=g.sort_values('band')
    assert np.all(g.lo.to_numpy()[1:]-g.hi.to_numpy()[:-1]>=.03-1e-6)
taxonomy=['vehicles','land_mammals','marine_animals','insects','musical_instruments','fruits','furniture','tools','plants','natural_phenomena']
cats=dict(zip(CFG['pair_ids'],['vehicles','fruits','furniture','musical_instruments','tools']))
targets={}
for pid in CFG['pair_ids']:
    g=cues[cues.pair==pid]
    assert g.x.nunique()==g.y.nunique()==1
    x,y=g.x.iloc[0],g.y.iloc[0]
    targets[pid]={'x':x,'original_y':y,'definition_x':defs[x],'definition_y':defs[y],'category':cats[pid]}
config={**CFG,'source_paths':paths,'input_sha256':{k:sha(b) for k,b in raw.items()},'generation_prompt':GEN,'candidate_audit_prompt':AUDIT,'cue_audit_prompt':CUE_AUDIT,'targets':targets,'taxonomy':taxonomy,'old_aliases':old['aliases']}
put('config.json',dump(config))
display(pd.DataFrame(targets).T)
print('Loaded 700 cues; old quality pass 622; X definitions and setting C bounds fixed.')


In [ ]:
# API checkpoint: one immutable start record per attempted call, one response record.
# Interrupted calls are not silently replayed under the same key.
def list_attempts():
    r=gh.get(API+DEST+'started',params={'ref':BRANCH},timeout=90)
    if r.status_code==404:return []
    r.raise_for_status();return sorted(j['name'] for j in r.json())
def ask(task,system,payload,validate):
    for attempt in range(CFG['max_attempts']):
        key=f'{task}_{attempt}'
        record=get('api/'+key+'.json')
        if record is not None:
            result=json.loads(record)
            if 'data' in result and validate(result['data']):return result['data']
            continue
        if get('started/'+key+'.json') is not None:
            print('Interrupted attempt:',key,flush=True);continue
        n=len(list_attempts())
        if n>=CFG['max_api_calls']:raise RuntimeError('API budget reached')
        put('started/'+key+'.json',dump({'task':task,'attempt':attempt,'system':system,'payload':payload}))
        print(f'API {n+1}/{CFG["max_api_calls"]}: {key}',flush=True)
        try:
            res=api.chat.completions.create(model=CFG['model'],reasoning_effort=CFG['reasoning_effort'],messages=[{'role':'system','content':system},{'role':'user','content':json.dumps(payload,ensure_ascii=False)}],response_format={'type':'json_object'})
            content=res.choices[0].message.content
            result={'raw_content':content,'model':res.model,'usage':res.usage.model_dump() if res.usage else {}}
            try:result['data']=json.loads(content)
            except (ValueError,TypeError):result['error']='invalid_json'
        except Exception as exc:
            result={'error':type(exc).__name__,'status_code':getattr(exc,'status_code',None)}
        put('api/'+key+'.json',dump(result))
        if 'data' in result and validate(result['data']):return result['data']
        if result.get('status_code') in (400,401,403,404):raise RuntimeError('API configuration/access failure; inspect status, do not substitute model silently.')
        time.sleep(3)
    raise RuntimeError('No valid response: '+task+'; records preserved; change run_id to retry exhausted tasks.')
def items_ok(d):
    return isinstance(d,dict) and isinstance(d.get('items'),list) and all(isinstance(i,dict) for i in d['items'])
def ids_ok(d,ids,fields):
    return items_ok(d) and len(d['items'])==len(ids) and {i.get('id') for i in d['items']}==set(ids) and all(fields(i) for i in d['items'])
groups=['same_close','same_medium','same_weak','different_related','different_weak']
def gen_ok(d):
    return items_ok(d) and len(d['items'])==40 and all(sum(i.get('group')==g for i in d['items'])==8 for g in groups) and all(all(isinstance(i.get(k),str) and i[k].strip() for k in ['concept','definition','category','group']) and isinstance(i.get('aliases'),list) and all(isinstance(a,str) for a in i['aliases']) for i in d['items'])
generated=[]
for pid,t in targets.items():
    data=ask('generate_'+pid,GEN,{**t,'taxonomy':taxonomy},gen_ok)
    for i,row in enumerate(data['items']):
        generated.append({'pair':pid,'candidate_id':f'{pid}_Y{i+1:02d}',**row})
assert len(generated)==200
put('generated_candidates.json',dump(generated))
# Independent concept audit (2 requests / X); audit sees no generation group or outcome.
audited=[]
for pid,t in targets.items():
    rows=[i for i in generated if i['pair']==pid]
    for start in range(0,len(rows),20):
        batch=rows[start:start+20]
        payload={'x':t['x'],'definition_x':t['definition_x'],'category_x':t['category'],'taxonomy':taxonomy,'items':[{'id':i['candidate_id'],'concept':i['concept'],'definition':i['definition']} for i in batch]}
        def fields(i):
            return i.get('verdict') in ('pass','fail','uncertain') and type(i.get('same_category')) is bool and isinstance(i.get('category'),str) and bool(i['category'].strip()) and isinstance(i.get('reason'),str) and isinstance(i.get('aliases'),list) and all(isinstance(a,str) for a in i['aliases'])
        data=ask(f'candidate_audit_{pid}_{start}',AUDIT,payload,lambda d:ids_ok(d,[i['candidate_id'] for i in batch],fields))
        mapping={i['id']:i for i in data['items']}
        for i in batch:
            a=mapping[i['candidate_id']]
            audited.append({**i,'audit_verdict':a['verdict'],'audit_category':a['category'],'same_category':a['same_category'],'audit_reason':a['reason'],'aliases':sorted(set(i['aliases']+a['aliases']))})
put('candidate_audit.json',dump(audited))
print('Generated and audited',len(audited),'Y candidates')


In [ ]:
# Fix all candidate eligibility before seeing any Cue-distance results.
tokenizer=AutoTokenizer.from_pretrained(CFG['tokenizer'],revision=CFG['tokenizer_revision'])
def tok(s):return len(tokenizer.encode(' '+s,add_special_tokens=False))
records=[]
for pid,t in targets.items():
    seen=set()
    for a in [z for z in audited if z['pair']==pid]:
        y=norm(a['concept']); definition=a['definition'].strip(); reasons=[]
        if y in {norm(t['x']),norm(t['original_y'])}|{norm(v) for v in old['aliases'].get(t['x'],[])+old['aliases'].get(t['original_y'],[])}:reasons.append('target_or_original_alias')
        if y in seen:reasons.append('duplicate_label')
        seen.add(y)
        if a['audit_verdict']!='pass':reasons.append('audit_'+a['audit_verdict'])
        if lexical(definition,[y]+a['aliases']):reasons.append('definition_name_exposure')
        if not 5<=len(definition.split())<=25 or '\n' in definition:reasons.append('definition_format')
        if not re.fullmatch(r"[a-z][a-z '\-]*",y):reasons.append('label_format')
        records.append({**a,'concept':y,'definition':definition,'x':t['x'],'definition_x':t['definition_x'],'category_x':t['category'],'original_y':False,'tokens_x':tok(t['x']),'tokens_y':tok(y),'exclusion_reason':';'.join(reasons),'eligible':not reasons})
    y=t['original_y']
    records.append({'pair':pid,'candidate_id':pid+'_anchor','concept':y,'definition':t['definition_y'],'category':t['category'],'group':'original_anchor','aliases':old['aliases'].get(y,[]),'audit_verdict':'historical_anchor_not_newly_audited','audit_category':t['category'],'same_category':True,'audit_reason':'original comparator','x':t['x'],'definition_x':t['definition_x'],'category_x':t['category'],'original_y':True,'tokens_x':tok(t['x']),'tokens_y':tok(y),'exclusion_reason':'','eligible':True})
cand=pd.DataFrame(records)
cand['token_diff']=(cand.tokens_y-cand.tokens_x).abs()
# Token-length mismatch is recorded, not excluded in exploratory B. Analyze subset <=1.
model=SentenceTransformer(CFG['embedding'],revision=CFG['revision'],device='cpu')
texts=sorted(set(cues.cue)|set(cand.definition)|set(cand.definition_x))
emb=model.encode(texts,normalize_embeddings=True,batch_size=32,show_progress_bar=True)
lookup=dict(zip(texts,emb))
def dist(a,b):return float(1-np.clip(np.dot(lookup[a],lookup[b]),-1,1))
cand['distance_xy']=[dist(a,b) for a,b in zip(cand.definition_x,cand.definition)]
cand['distance_band']=-1; cand['band_center']=np.nan; cand['primary_selected']=False
# New candidates only, rank-balanced empirical quintiles, choose nearest median 3;
# original comparator always separate anchor, not counted in primary 75.
for pid,g in cand[cand.eligible & ~cand.original_y].groupby('pair'):
    order=g.sort_values(['distance_xy','concept','candidate_id']).index.to_numpy()
    for band,idx in enumerate(np.array_split(order,5)):
        if not len(idx):continue
        center=float(cand.loc[idx,'distance_xy'].median())
        cand.loc[idx,'distance_band']=band; cand.loc[idx,'band_center']=center
        chosen=cand.loc[idx].assign(delta=lambda z:abs(z.distance_xy-center)).sort_values(['delta','concept','candidate_id']).head(3).index
        cand.loc[chosen,'primary_selected']=True
cand['analysis_selected']=cand.primary_selected|cand.original_y
cand['aliases_json']=cand.aliases.map(lambda a:json.dumps(a,ensure_ascii=False))
save_df('candidate_pairs.csv',cand.drop(columns='aliases'))
save_df('concept_definitions.csv',cand[['candidate_id','pair','x','definition_x','concept','definition','aliases_json','eligible','original_y']])
save_df('xy_distance.csv',cand[['pair','candidate_id','x','concept','distance_xy','distance_band','band_center','primary_selected','original_y','eligible','same_category','token_diff']])
selected=cand[cand.analysis_selected].copy()
coverage=cand.groupby('pair').agg(generated=('original_y',lambda x:(~x).sum()),eligible=('eligible','sum'),primary=('primary_selected','sum'),anchors=('original_y','sum')).reset_index()
coverage['eligible_new']=coverage.eligible-coverage.anchors
save_df('candidate_coverage.csv',coverage)
display(coverage);display(selected[['pair','concept','distance_xy','distance_band','same_category','token_diff']])
print('Primary selected:',selected.primary_selected.sum(),'anchors:',selected.original_y.sum())


In [ ]:
# Common Cue sets: old measured dX is reused exactly and checked against recomputation.
# dY is recomputed for every eligible Y. Name exposure is recorded, not used to alter main sample.
cue_frames=[]
for pid,t in targets.items():
    g=cues[cues.pair==pid].copy()
    g['d_x_recomputed']=[dist(t['definition_x'],s) for s in g.cue]
    assert np.max(abs(g.d_x-g.d_x_recomputed))<2e-4,'Embedding mismatch'
    g['fixed_band']=-1
    for b in bounds[bounds.pair==pid].itertuples():
        g.loc[(g.related==1)&g.d_x.between(b.lo,b.hi),'fixed_band']=int(b.band)
    g.loc[g.related==0,'fixed_band']=5
    for a in cand[(cand.pair==pid)&cand.eligible].itertuples():
        z=g[['id','pair','x','cue','style','intended','related','quality_pass','d_x','fixed_band']].copy()
        z['candidate_id']=a.candidate_id;z['y']=a.concept
        z['d_y']=[dist(a.definition,s) for s in z.cue]
        z['y_name_exposed']=[lexical(s,[a.concept]+a.aliases) for s in z.cue]
        if a.original_y:assert np.max(abs(z.d_y.to_numpy()-g.d_y.to_numpy()))<2e-4
        cue_frames.append(z)
measured=pd.concat(cue_frames,ignore_index=True)
save_df('cue_distances_all_eligible.csv',measured)
print('Measured candidate × Cue rows:',len(measured))
# Common audit sample of old quality-pass Cue IDs, stratified by original intended band.
sample_ids={}
for pid,g in cues[cues.quality_pass].groupby('pair'):
    parts=[]
    for band,h in g.groupby('intended'):
        h=h.sort_values('id')
        n=len(h) if CFG['semantic_audit_mode']=='full' else min(CFG['semantic_audit_per_intended_band'],len(h))
        if CFG['semantic_audit_mode']!='off':parts.extend(h.sample(n=n,random_state=CFG['seed']).id.tolist())
    sample_ids[pid]=sorted(parts)
put('cue_audit_sample_ids.json',dump(sample_ids))
print('Semantic audit sample size per X:',{k:len(v) for k,v in sample_ids.items()})


In [ ]:
# Optional semantic audit: same sampled cues for every selected Y and anchor.
# Default sample is not a full quality-screened corpus; report both sample before/after filtering.
semantic=[]
for a in selected.itertuples():
    g=measured[(measured.candidate_id==a.candidate_id)&measured.id.isin(sample_ids[a.pair])].sort_values('id')
    for start in range(0,len(g),CFG['semantic_audit_batch_size']):
        h=g.iloc[start:start+CFG['semantic_audit_batch_size']]
        payload={'x':a.x,'definition_x':a.definition_x,'y':a.concept,'definition_y':a.definition,'aliases_x':old['aliases'].get(a.x,[]),'aliases_y':a.aliases,'items':h[['id','cue']].to_dict('records')}
        def fields(i):
            return all(type(i.get(k)) is int and 0<=i[k]<=4 for k in ['x_related','y_related']) and i.get('exposure') in ('pass','fail','uncertain') and i.get('y_specific') in ('yes','no','uncertain') and isinstance(i.get('reason'),str)
        data=ask(f'cue_audit_{a.candidate_id}_{start}',CUE_AUDIT,payload,lambda d:ids_ok(d,h.id.tolist(),fields))
        semantic.extend({'candidate_id':a.candidate_id,**i} for i in data['items'])
sem=pd.DataFrame(semantic,columns=['candidate_id','id','x_related','y_related','exposure','y_specific','reason'])
assert not sem.duplicated(['candidate_id','id']).any()
save_df('cue_semantic_audit.csv',sem)
print('Semantically audited rows:',len(sem))


In [ ]:
# Metrics: related and unrelated never pooled for primary correlations.
# Fixed bands are original v3 setting C, not redefined when Y changes.
def corr(a,b,fn):
    a=np.asarray(a);b=np.asarray(b)
    if len(a)<3 or np.ptp(a)==0 or np.ptp(b)==0:return np.nan
    return float(fn(a,b).statistic)
def summary(g):
    bands=g[g.fixed_band.between(0,4)].groupby('fixed_band').d_y.mean()
    return {'n':len(g),'pearson':corr(g.d_x,g.d_y,pearsonr),'spearman':corr(g.d_x,g.d_y,spearmanr),'abs_spearman':abs(corr(g.d_x,g.d_y,spearmanr)),'dy_mean':g.d_y.mean(),'dy_sd':g.d_y.std(),'band_mean_spread':bands.max()-bands.min() if len(bands)==5 else np.nan,'bands_present':len(bands),'name_exposure_n':int(g.y_name_exposed.sum())}
metrics=[];bandrows=[];slots=[]
# Near-duplicate constraints mirror v4. Precompute once per X.
conflicts={}
for pid,g in cues.groupby('pair'):
    g=g.sort_values('id')
    conflicts[pid]=[(a.id,b.id) for a,b in itertools.combinations(list(g.itertuples()),2) if SequenceMatcher(None,norm(a.cue),norm(b.cue)).ratio()>=.80]
styles=['structure','action','situation','function']
def feasibility(g):
    g=g[g.fixed_band>=0].reset_index(drop=True)
    lookup_idx={v:i for i,v in enumerate(g.id)}
    pools=[((g.fixed_band==b)&(g['style']==s)).to_numpy(dtype=float) for b,s in itertools.product(range(6),styles)]
    if not len(g):return 0,0
    matrix=list(pools)
    for a,b in conflicts[g.pair.iloc[0]]:
        if a in lookup_idx and b in lookup_idx:
            row=np.zeros(len(g));row[lookup_idx[a]]=row[lookup_idx[b]]=1;matrix.append(row)
    result=milp(-np.ones(len(g)),integrality=np.ones(len(g)),bounds=Bounds(0,1),constraints=LinearConstraint(np.array(matrix),0,1),options={'time_limit':30})
    return int(sum(p.any() for p in pools)),int(np.rint(result.x).sum()) if result.success else np.nan
for a in cand[cand.eligible].itertuples():
    g=measured[measured.candidate_id==a.candidate_id]
    semg=g.merge(sem[sem.candidate_id==a.candidate_id],on=['candidate_id','id'],how='inner',validate='one_to_one')
    if len(semg):
        semg['semantic_eligible']=(semg.exposure=='pass')&(semg.y_specific=='no')&(~semg.y_name_exposed)&(((semg.related==1)&(semg.x_related>=1))|((semg.related==0)&(semg.x_related==0)&(semg.y_related==0)))
    else:semg['semantic_eligible']=False
    subsets={'all_common':g,'old_quality_common':g[g.quality_pass],'old_quality_no_y_name':g[g.quality_pass&~g.y_name_exposed]}
    if len(semg):subsets.update({'semantic_audit_common_sample':semg,'semantic_audit_filtered_sample':semg[semg.semantic_eligible]})
    for label,h in subsets.items():
        for rel in (1,0):
            q=h[h.related==rel]
            metrics.append({'candidate_id':a.candidate_id,'pair':a.pair,'x':a.x,'y':a.concept,'subset':label,'related':rel,**summary(q)})
        for b,k in h[h.fixed_band>=0].groupby('fixed_band'):
            bandrows.append({'candidate_id':a.candidate_id,'pair':a.pair,'subset':label,'band':b,'n':len(k),'dy_mean':k.d_y.mean(),'dy_sd':k.d_y.std(),'dx_mean':k.d_x.mean()})
    # Structural availability, not joint-Y optimization. Only selected candidates + anchors.
    if a.analysis_selected:
        for label in ['old_quality_common','old_quality_no_y_name']+(['semantic_audit_filtered_sample'] if CFG['semantic_audit_mode']=='full' else []):
            h=subsets.get(label,g.iloc[:0])
            filled,max_n=feasibility(h)
            for b,s in itertools.product(range(6),styles):
                q=h[(h.fixed_band==b)&(h['style']==s)]
                slots.append({'candidate_id':a.candidate_id,'pair':a.pair,'subset':label,'band':b,'style':s,'n':len(q),'dy_min':q.d_y.min(),'dy_max':q.d_y.max(),'nonempty_slots':filled,'maximum_nonduplicate_n':max_n,'structurally_complete':max_n==24})
meta=cand[['candidate_id','distance_xy','distance_band','same_category','token_diff','primary_selected','original_y','analysis_selected']]
metrics=pd.DataFrame(metrics).merge(meta,on='candidate_id',validate='many_to_one')
bandrows=pd.DataFrame(bandrows);slots=pd.DataFrame(slots)
save_df('cue_distance_correlations.csv',metrics)
save_df('cue_band_dy_summary.csv',bandrows)
save_df('cue_slot_inventory.csv',slots)
if len(slots):
    slot_summary=slots[['candidate_id','pair','subset','nonempty_slots','maximum_nonduplicate_n','structurally_complete']].drop_duplicates().merge(meta,on='candidate_id',validate='many_to_one')
    save_df('cue_structural_feasibility.csv',slot_summary)
    display(slot_summary.groupby('subset').agg(pairs=('candidate_id','size'),complete=('structurally_complete','sum'),mean_slots=('maximum_nonduplicate_n','mean')))


In [ ]:
# Within-X analysis. No pooled nominal p-values, since candidates share X and cues.
# Primary: selected new Y, common old-quality related cues. All common and all eligible sensitivity.
relations=[];comparisons=[]
for subset in ['all_common','old_quality_common','old_quality_no_y_name','semantic_audit_common_sample','semantic_audit_filtered_sample']:
    h=metrics[(metrics.subset==subset)&(metrics.related==1)&~metrics.original_y]
    for pop,q in [('selected',h[h.primary_selected]),('all_eligible',h)]:
        for pid,g in q.groupby('pair'):
            for stratum,z in [('all',g),('same_category',g[g.same_category]),('different_category',g[~g.same_category]),('token_diff_le1',g[g.token_diff<=1])]:
                for outcome in ['spearman','abs_spearman','band_mean_spread']:
                    a=z[['distance_xy',outcome]].dropna()
                    relations.append({'subset':subset,'population':pop,'pair':pid,'stratum':stratum,'outcome':outcome,'n':len(a),'pearson':corr(a.distance_xy,a[outcome],pearsonr),'spearman':corr(a.distance_xy,a[outcome],spearmanr),'distance_min':a.distance_xy.min(),'distance_max':a.distance_xy.max()})
        if pop=='selected':
            for (pid,b),g in q.groupby(['pair','distance_band']):
                comparisons.append({'subset':subset,'pair':pid,'distance_band':b,'n':len(g),'mean_distance':g.distance_xy.mean(),'mean_abs_rho':g.abs_spearman.mean(),'mean_spread':g.band_mean_spread.mean(),'same_category_n':int(g.same_category.sum())})
rel=pd.DataFrame(relations);comparison=pd.DataFrame(comparisons)
save_df('within_x_associations.csv',rel);save_df('distance_band_comparison.csv',comparison)
primary=rel[(rel.subset=='old_quality_common')&(rel.population=='selected')&(rel.stratum=='all')]
display(primary)
signs=primary.groupby('outcome').agg(n_x=('spearman','count'),negative_x=('spearman',lambda s:(s<0).sum()),median_spearman=('spearman','median')).reset_index()
save_df('within_x_direction_summary.csv',signs);display(signs)
# Explicit overlapping-distance category comparison: restrict each X to common support.
overlap=[]
for pid,g in metrics[(metrics.subset=='old_quality_common')&(metrics.related==1)&metrics.primary_selected].groupby('pair'):
    s=g[g.same_category];d=g[~g.same_category]
    if s.empty or d.empty:continue
    lo=max(s.distance_xy.min(),d.distance_xy.min());hi=min(s.distance_xy.max(),d.distance_xy.max())
    for same,h in g[g.distance_xy.between(lo,hi)].groupby('same_category'):
        overlap.append({'pair':pid,'same_category':same,'support_lo':lo,'support_hi':hi,'n':len(h),'mean_abs_rho':h.abs_spearman.mean(),'mean_spread':h.band_mean_spread.mean()})
save_df('category_overlap_descriptive.csv',pd.DataFrame(overlap,columns=['pair','same_category','support_lo','support_hi','n','mean_abs_rho','mean_spread']))


In [ ]:
# Figures and final report.
outputs=['candidate_pairs.csv','concept_definitions.csv','xy_distance.csv','candidate_coverage.csv','cue_distances_all_eligible.csv','cue_semantic_audit.csv','cue_distance_correlations.csv','cue_band_dy_summary.csv','cue_slot_inventory.csv','cue_structural_feasibility.csv','within_x_associations.csv','distance_band_comparison.csv','within_x_direction_summary.csv','category_overlap_descriptive.csv','config.json','generated_candidates.json','candidate_audit.json','cue_audit_sample_ids.json']
def figsave(name,fig):
    buf=io.BytesIO();fig.savefig(buf,format='png',dpi=150,bbox_inches='tight')
    path='figures/'+name+'.png';put(path,buf.getvalue());outputs.append(path);plt.show();plt.close(fig)
p=metrics[(metrics.subset=='old_quality_common')&(metrics.related==1)&metrics.analysis_selected]
for outcome in ['abs_spearman','band_mean_spread']:
    fig,axs=plt.subplots(1,5,figsize=(17,3.7),sharey=True)
    for ax,(pid,t) in zip(axs,targets.items()):
        h=p[p.pair==pid]
        for same,marker,label in [(True,'o','same category'),(False,'^','different category')]:
            q=h[(h.same_category==same)&~h.original_y]
            ax.scatter(q.distance_xy,q[outcome],marker=marker,label=label,alpha=.8)
        q=h[h.original_y];ax.scatter(q.distance_xy,q[outcome],marker='*',s=110,label='original Y')
        ax.axhline(.30 if outcome=='abs_spearman' else .05,ls='--',color='gray')
        ax.set_title(t['x']);ax.set_xlabel('X/Y distance')
    axs[0].set_ylabel(outcome);axs[-1].legend(fontsize=7);fig.tight_layout();figsave(outcome,fig)
fig,axs=plt.subplots(1,5,figsize=(17,3.5),sharey=True)
for ax,(pid,t) in zip(axs,targets.items()):
    g=cand[(cand.pair==pid)&cand.eligible&~cand.original_y]
    ax.hist(g.distance_xy,bins=10,alpha=.7)
    q=g[g.primary_selected];ax.scatter(q.distance_xy,np.zeros(len(q)),marker='|',s=100)
    ax.set_title(t['x']);ax.set_xlabel('X/Y distance')
axs[0].set_ylabel('Eligible Y candidates');fig.tight_layout();figsave('candidate_distance_histograms',fig)
usage=[]
for n in list_attempts():
    rec=get('api/'+n)
    a=json.loads(rec) if rec else {}
    u=a.get('usage',{})
    usage.append({'attempt':n,'model':a.get('model'),'error':a.get('error','' if rec else 'interrupted'),'prompt_tokens':u.get('prompt_tokens'),'completion_tokens':u.get('completion_tokens'),'total_tokens':u.get('total_tokens')})
usage=pd.DataFrame(usage);save_df('api_usage.csv',usage);outputs.append('api_usage.csv')
report=['# Experiment B — exploratory results','',
 f"Generated 200 Y candidates; eligible new {int((cand.eligible&~cand.original_y).sum())}; primary selected {int(cand.primary_selected.sum())}/75; original anchors {int(cand.original_y.sum())}.",
 '700 original cues, 622 historical quality pass; X definitions and dX fixed; related and unrelated separated.',
 'Primary: old-quality common related cues, selected new Y, within-X correlations. All-cue and all-eligible analyses are sensitivity analyses.',
 f"Semantic audit mode: {CFG['semantic_audit_mode']}; audited rows: {len(sem)}. Sample mode does not certify full cue-set quality.",
 'No new Cue generation, Qwen forward passes, Probe learning or joint optimization. Slot feasibility is structural (distance, style, surface deduplication), NOT success at Y-distance control.',
 '', '## Within-X Spearman association between X/Y distance and each outcome']
for z in primary.itertuples():report.append(f"- {z.pair} / {z.outcome}: rho={z.spearman:.4f}, n={z.n}")
report+=['','## Limitations','Candidates share X and Cue sets, so 75 pairs are not 75 independent replicates. Category and token length may confound distance; overlap and restricted analyses are descriptive. Old cues were generated/audited with original Y, which can favor anchors. Candidate definitions and semantic judgments are AI-produced and not human-verified. No causal or formal threshold claim. No suppression results used.']
put('analysis_summary.md',('\n'.join(report)+'\n').encode());outputs.append('analysis_summary.md')
manifest={'status':'complete_exploratory','config_sha256':sha(dump(config)),'input_sha256':config['input_sha256'],'source_ref':CFG['source_ref'],'source_A_ref':CFG['source_A_ref'],'seed':CFG['seed'],'embedding_revision':CFG['revision'],'tokenizer_revision':CFG['tokenizer_revision'],'versions':{k:im.version(k) for k in ['openai','sentence-transformers','transformers','torch','numpy','pandas','scipy']},'python':platform.python_version(),'n_primary':int(cand.primary_selected.sum()),'n_anchors':int(cand.original_y.sum()),'n_api_attempts':len(usage),'files':{n:sha(get(n)) for n in outputs}}
put('manifest.json',dump(manifest))
print('\n'.join(report))
print('Results: https://github.com/'+REPO+'/tree/'+BRANCH+'/'+DEST)
